# RCO-Enhanced NeuroCapsNet for Speech-Based Parkinson’s Disease Classification

This notebook implements the manuscript pipeline: mean imputation → modified Box–Cox transform → Improved LDA → statistical/correlation/Gini features + AGC-LSTM representation → RemoCham (ROA–CSA) binary feature selection → MLP + two-layer Bi-LSTM + temporal capsule routing (NeuroCapsNet).

**Evaluation is subject-wise**: all recordings from a participant remain in one fold. All learned preprocessing, feature extraction, feature selection and model fitting are performed inside each training fold.

> This is a research implementation for binary dataset classification, not a clinical diagnostic device. Reported manuscript metrics are targets/results from the paper, not guaranteed outputs of a new run. Database 1 and Database 2 are described as duplicate copies of the Oxford dataset and should not be treated as independent evidence.

## 1. Environment

Run this notebook in Google Colab or a Python environment with internet access. The code uses PyTorch and scikit-learn. Kaggle downloads are optional: you can upload a CSV manually or set the paths below.

In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn torch tqdm joblib

In [ ]:
import os, re, glob, math, time, random, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis, gmean, hmean, mode, entropy
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, classification_report)
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Dataset configuration and loading

The manuscript describes:
- **DB1 / DB2**: Oxford Parkinson’s voice dataset, 195 recordings, 31 participants, 22 acoustic measures; `status` is the label and `name` contains participant/recording identity.
- **DB3**: 756 recordings, 252 participants, 754 speech-derived features; retain the dataset’s participant identifier for grouping.

Set `DATASET_PATHS` to local CSV files. In Colab, upload files to `/content/` or mount Google Drive. Do not merge DB1 and DB2 because the manuscript says they are the same underlying data.

In [ ]:
# EDIT THESE PATHS. Example: (participant IDs must be retained for grouped CV)
# DATASET_PATHS = {
#   "DB1": "/content/parkinsons.csv",
#   "DB3": "/content/parkinsons_speech_features.csv"
# }
DATASET_PATHS = {
    "DB1": "/content/parkinsons.csv",
    "DB3": "/content/parkinsons_speech_features.csv"
}

# If paths do not exist, this cell offers a Colab upload dialog.
def locate_csvs(paths):
    found = {k:v for k,v in paths.items() if os.path.exists(v)}
    if not found:
        try:
            from google.colab import files
            print("Upload one or more dataset CSV files:")
            uploaded = files.upload()
            for name in uploaded:
                if name.lower().endswith(".csv"):
                    key = "DB3" if any(x in name.lower() for x in ["speech", "signal", "754"]) else "DB1"
                    found[key] = "/content/" + name
        except Exception:
            pass
    return found

DATASET_PATHS = locate_csvs(DATASET_PATHS)
print(DATASET_PATHS)

In [ ]:
def infer_label_column(df):
    candidates = ["status", "label", "class", "target", "diagnosis", "parkinson", "health_status"]
    lower = {c.lower():c for c in df.columns}
    for c in candidates:
        if c in lower: return lower[c]
    # fallback: binary-valued object/category or integer columns near the end
    for c in reversed(df.columns):
        vals = df[c].dropna().unique()
        if len(vals) == 2 and (df[c].dtype == "object" or set(map(str, vals)).issubset({"0","1","0.0","1.0"})):
            return c
    raise ValueError("Could not infer label column. Set LABEL_COLUMN manually.")

def infer_group_column(df, label_col):
    candidates = ["subject_id", "participant_id", "patient_id", "subject", "participant", "name", "recording_id", "id"]
    lower = {c.lower():c for c in df.columns}
    for c in candidates:
        if c in lower and lower[c] != label_col: return lower[c]
    return None

def normalize_binary_labels(y):
    s = pd.Series(y)
    if pd.api.types.is_numeric_dtype(s):
        vals = sorted(s.dropna().unique())
        if len(vals) != 2: raise ValueError(f"Expected 2 classes, found {vals}")
        return s.map({vals[0]:0, vals[1]:1}).astype(int).to_numpy()
    z = s.astype(str).str.lower().str.strip()
    positive = z.str.contains(r"parkinson|\bpd\b|positive|patient|disease|1", regex=True)
    negative = z.str.contains(r"healthy|control|normal|negative|0", regex=True)
    if (positive & negative).any() or not (positive | negative).all():
        le=LabelEncoder(); return le.fit_transform(z)
    return positive.astype(int).to_numpy()

def derive_groups(df, group_col):
    if group_col is None:
        warnings.warn("No participant/group column found. Using each row as a group is NOT subject-independent validation.")
        return np.arange(len(df)).astype(str)
    raw=df[group_col].astype(str).fillna("missing")
    # Oxford 'name' values commonly contain recording suffixes; strip trailing recording index.
    if group_col.lower()=="name":
        raw=raw.str.replace(r"[_\- ]?\d+$","",regex=True)
    return raw.to_numpy()

def load_dataset(path, name):
    df=pd.read_csv(path)
    label_col=infer_label_column(df)
    group_col=infer_group_column(df,label_col)
    y=normalize_binary_labels(df[label_col])
    groups=derive_groups(df,group_col)
    exclude={label_col}
    if group_col: exclude.add(group_col)
    # Keep numeric predictors only; identifiers and nonnumeric metadata are excluded.
    X=df.drop(columns=list(exclude),errors="ignore").select_dtypes(include=[np.number]).copy()
    X=X.replace([np.inf,-np.inf],np.nan)
    if X.shape[1] == 0: raise ValueError(f"{name}: no numeric feature columns found.")
    # Remove constant/all-missing columns; imputer is still fit inside folds below.
    keep=X.columns[X.notna().any() & (X.nunique(dropna=True)>1)]
    X=X[keep]
    print(f"{name}: rows={len(df)}, numeric_features={X.shape[1]}, groups={len(np.unique(groups))}, label={label_col}, group={group_col}")
    print("Class counts:", dict(zip(*np.unique(y,return_counts=True))))
    return X.to_numpy(dtype=np.float32), y.astype(int), groups, list(X.columns), df

datasets={}
for name,path in DATASET_PATHS.items():
    datasets[name]=load_dataset(path,name)
if not datasets:
    raise FileNotFoundError("No CSV loaded. Set DATASET_PATHS to your downloaded dataset CSV files and rerun this cell.")

## 3. Modified Box–Cox and Improved LDA

The modified Box–Cox step shifts each feature to positive support using a training-fold minimum and applies a Box–Cox transform; near-constant columns are safely retained as zero. I-LDA uses class-balanced between-class scatter, regularized within-class scatter, discriminant-ratio feature pre-ranking, eigenvector projection and centroid-derived bias. The transformation is fit only on training data.

In [ ]:
class ModifiedBoxCox(BaseEstimator, TransformerMixin):
    def __init__(self, eps=1e-6): self.eps=eps
    def fit(self, X, y=None):
        from scipy.stats import boxcox_normmax
        X=np.asarray(X,dtype=float)
        self.min_=np.nanmin(X,axis=0)
        self.shift_=np.maximum(0, -self.min_ + self.eps)
        shifted=X+self.shift_
        self.lambdas_=[]
        for j in range(shifted.shape[1]):
            col=np.maximum(shifted[:,j],self.eps)
            if np.nanstd(col)<1e-10:
                self.lambdas_.append(1.0)
            else:
                try: self.lambdas_.append(float(boxcox_normmax(col,method="mle")))
                except Exception: self.lambdas_.append(0.0)
        return self
    def transform(self, X):
        from scipy.special import boxcox
        X=np.asarray(X,dtype=float)+self.shift_
        out=np.zeros_like(X)
        for j,lam in enumerate(self.lambdas_):
            col=np.maximum(X[:,j],self.eps)
            if np.nanstd(col)<1e-10: out[:,j]=0
            else:
                try: out[:,j]=boxcox(col,lam)
                except Exception: out[:,j]=np.log(col)
        return np.nan_to_num(out,nan=0,posinf=0,neginf=0)

class ImprovedLDA(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=1, reg=1e-3, max_features=256):
        self.n_components=n_components; self.reg=reg; self.max_features=max_features
    def fit(self,X,y):
        X=np.asarray(X,float); y=np.asarray(y)
        self.classes_=np.unique(y)
        if len(self.classes_)!=2: raise ValueError("I-LDA implementation expects binary labels.")
        # Discriminant-ratio pre-ranking
        means=np.array([X[y==c].mean(0) for c in self.classes_])
        vari=np.array([X[y==c].var(0) for c in self.classes_])
        score=(means[1]-means[0])**2/(vari.mean(0)+self.reg)
        k=min(X.shape[1],self.max_features)
        self.idx_=np.argsort(score)[::-1][:k]
        Z=X[:,self.idx_]
        means=np.array([Z[y==c].mean(0) for c in self.classes_])
        Sw=np.zeros((k,k))
        for c,m in zip(self.classes_,means):
            A=Z[y==c]-m
            Sw += A.T@A
        Sw=Sw/max(len(y),1)+self.reg*np.eye(k)
        overall=Z.mean(0)
        # Equal class contribution to between-class scatter
        Sb=np.zeros((k,k))
        for m in means:
            d=(m-overall).reshape(-1,1)
            Sb += 0.5*(d@d.T)
        # Stable generalized eigenproblem via whitening
        vals,U=np.linalg.eigh(Sw)
        W=(U*(1/np.sqrt(np.maximum(vals,self.reg))))@U.T
        M=W@Sb@W
        ev,V=np.linalg.eigh(M)
        order=np.argsort(ev)[::-1]
        self.proj_=W@V[:,order[:min(self.n_components,len(order))]]
        self.centroids_=np.array([(Z[y==c]@self.proj_).mean(0) for c in self.classes_])
        self.bias_=-0.5*(self.centroids_[0]+self.centroids_[1])
        return self
    def transform(self,X):
        Z=np.asarray(X,float)[:,self.idx_]
        return (Z@self.proj_ + self.bias_).astype(np.float32)

## 4. Complementary feature representations and AGC-LSTM

For every recording, the handcrafted branch computes statistical descriptors, Pearson-correlation summaries and Gini impurity. The learned branch constructs a feature-correlation graph and applies multi-head graph attention followed by an LSTM. For tabular speech descriptors, the sequence axis is the ordered feature vector; it is not a raw waveform.

In [ ]:
def handcrafted_features(Z, corr_stats=None):
    Z=np.asarray(Z,float)
    n,d=Z.shape
    out=[]
    for row in Z:
        a=np.asarray(row,float); pos=np.abs(a)+1e-6
        vals=[
            np.mean(a), np.std(a), np.median(a), np.var(a),
            skew(a) if len(a)>2 else 0, kurtosis(a) if len(a)>3 else 0,
            np.max(a)-np.min(a), np.sqrt(np.mean(a*a)),
            np.mean(np.abs(a)), np.sum(a*a)/(len(a)+1e-8),
            np.exp(np.mean(np.log(pos))), len(a)/(np.sum(1/pos)+1e-8)
        ]
        out.append(vals)
    H=np.asarray(out,dtype=np.float32)
    # Pearson correlation summary is estimated from training data only.
    if corr_stats is None:
        C=np.corrcoef(Z,rowvar=False) if Z.shape[1]>1 else np.ones((1,1))
        C=np.nan_to_num(C)
        corr_stats=np.array([np.mean(np.abs(C-np.eye(C.shape[0]))),
                             np.max(np.abs(C-np.eye(C.shape[0]))),
                             np.mean(C), np.std(C)],dtype=np.float32)
    corr=np.tile(np.asarray(corr_stats,dtype=np.float32),(n,1))
    # Gini uncertainty from labels is added separately by the training labels below.
    return np.concatenate([H,corr],axis=1)

def gini_feature(y, n):
    p=np.bincount(np.asarray(y,dtype=int),minlength=2).astype(float)
    p=p/max(p.sum(),1)
    g=1-np.sum(p*p)
    return np.full((n,1),g,dtype=np.float32)

class AGCLSTM(nn.Module):
    def __init__(self, in_features, hidden=32, heads=4, out_dim=16):
        super().__init__()
        self.proj=nn.Linear(1,hidden)
        self.attn=nn.MultiheadAttention(hidden,heads,batch_first=True,dropout=0.2)
        self.norm=nn.LayerNorm(hidden)
        self.lstm=nn.LSTM(hidden,hidden//2,batch_first=True,bidirectional=True)
        self.fc=nn.Sequential(nn.Linear(hidden,out_dim),nn.ReLU(),nn.Dropout(.25))
    def forward(self,x):
        # x: [batch, feature_count]; each feature is a graph node.
        h=self.proj(x.unsqueeze(-1))
        a,_=self.attn(h,h,h,need_weights=False)
        h=self.norm(h+a)
        h,_=self.lstm(h)
        return self.fc(h.mean(dim=1))

def train_agc(trainX, trainY, applyX, epochs=12, batch_size=64):
    model=AGCLSTM(trainX.shape[1]).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    loss_fn=nn.CrossEntropyLoss()
    ds=TensorDataset(torch.tensor(trainX,dtype=torch.float32),torch.tensor(trainY,dtype=torch.long))
    dl=DataLoader(ds,batch_size=min(batch_size,len(ds)),shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb,yb in dl:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(); logits=model(xb); loss=loss_fn(logits,yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),2.0); opt.step()
    model.eval()
    with torch.no_grad():
        z=model(torch.tensor(applyX,dtype=torch.float32).to(DEVICE)).cpu().numpy()
    return z,model

## 5. RemoCham Optimizer (RCO) feature selection

Binary agents encode selected features. The hybrid search combines CSA-style global movement with ROA-style refinement around the best solution. Fitness balances validation classification error, feature redundancy and subset size. The fitness validation split is group-aware and uses training-fold data only.

In [ ]:
def subset_fitness(mask, X, y, groups, alpha=.70, beta=.15, gamma=.15):
    idx=np.where(mask)[0]
    if len(idx)==0: return 1e6
    # group-aware holdout where possible
    try:
        splitter=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED)
        tr,va=next(splitter.split(X,y,groups))
        if len(np.unique(y[tr]))<2 or len(np.unique(y[va]))<2: raise ValueError()
    except Exception:
        from sklearn.model_selection import StratifiedShuffleSplit
        tr,va=next(StratifiedShuffleSplit(n_splits=1,test_size=.25,random_state=SEED).split(X,y))
    clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=400,class_weight="balanced"))
    try:
        clf.fit(X[tr][:,idx],y[tr])
        err=1-accuracy_score(y[va],clf.predict(X[va][:,idx]))
    except Exception: err=1.0
    if len(idx)>1:
        C=np.corrcoef(X[:,idx],rowvar=False); C=np.nan_to_num(C)
        red=np.sum(np.abs(C-np.eye(len(idx))))/(len(idx)*(len(idx)-1))
    else: red=0.0
    compact=len(idx)/X.shape[1]
    return alpha*err + beta*red + gamma*compact

def rco_select(X,y,groups,population=12,iterations=15,seed=42):
    rng=np.random.default_rng(seed); n,d=X.shape
    pop=rng.uniform(0,1,(population,d))
    # sparsity-aware initialization
    pop[:, :] = np.where(pop < .35, 0.0, pop)
    def decode(p): return (1/(1+np.exp(-8*(p-.5))) > rng.random(d))
    masks=np.array([decode(p) for p in pop])
    masks[:,0]=True
    fit=np.array([subset_fitness(m,X,y,groups) for m in masks])
    best=masks[np.argmin(fit)].copy(); bestfit=float(fit.min())
    curve=[bestfit]
    for it in range(iterations):
        frac=it/max(iterations-1,1)
        new=[]
        for i in range(population):
            if i%2==0:
                # ROA-inspired exploitation: host-following/local refinement
                candidate=pop[i] + rng.uniform(-.25,.25,d)*(best.astype(float)-pop[i])
            else:
                # CSA-inspired exploration: broad prey/visual-search movement
                peer=pop[rng.integers(population)]
                candidate=pop[i] + rng.uniform(-1,1,d)*(peer-pop[i]) + rng.normal(0,.15,d)*(1-frac)
            candidate=np.clip(candidate,0,1)
            mask=(candidate>rng.uniform(.35,.65,d))
            if not mask.any(): mask[np.argmax(candidate)]=True
            new.append((candidate,mask))
        for i,(p,m) in enumerate(new):
            f=subset_fitness(m,X,y,groups)
            if f<fit[i]: pop[i]=p; masks[i]=m; fit[i]=f
            if f<bestfit: best=m.copy(); bestfit=float(f)
        # exchange best candidate with worst candidate to encourage cooperation
        worst=np.argmax(fit); pop[worst]=best.astype(float); masks[worst]=best.copy(); fit[worst]=bestfit
        curve.append(bestfit)
    return np.where(best)[0],curve

## 6. NeuroCapsNet: MLP + two-layer Bi-LSTM + temporal capsule routing

The selected vector is embedded by an MLP, reshaped into a short sequence, processed by a two-layer bidirectional LSTM and converted into primary temporal capsules. Dynamic routing aggregates them into two class capsules.

In [ ]:
def squash(v, dim=-1, eps=1e-8):
    s=(v*v).sum(dim=dim,keepdim=True)
    scale=s/(1+s)/torch.sqrt(s+eps)
    return scale*v

class NeuroCapsNet(nn.Module):
    def __init__(self,input_dim,hidden=64,caps_dim=16,routing_iters=3):
        super().__init__()
        self.routing_iters=routing_iters
        self.mlp=nn.Sequential(nn.Linear(input_dim,128),nn.ReLU(),nn.Dropout(.30),
                               nn.Linear(128,hidden),nn.ReLU())
        self.bilstm=nn.LSTM(hidden,hidden//2,num_layers=2,batch_first=True,
                            bidirectional=True,dropout=.30)
        self.primary=nn.Linear(hidden,caps_dim)
        self.W=nn.Parameter(torch.randn(1,2,1,caps_dim,caps_dim)*.05)
    def forward(self,x):
        h=self.mlp(x)
        # Preserve ordered feature representation as a short temporal sequence.
        seq=h.unsqueeze(1).repeat(1,8,1)
        seq, _=self.bilstm(seq)
        u=squash(self.primary(seq))
        # [B, classes, time, capsule_dim]
        uhat=torch.einsum("btd,ocde->bo t e",u,self.W.squeeze(0)) if False else None
        # class-specific capsule transforms
        W=self.W.expand(x.size(0),-1,u.size(1),-1,-1)
        votes=torch.einsum("btd,botde->bote",u,W)
        b=torch.zeros(x.size(0),2,u.size(1),device=x.device)
        for _ in range(self.routing_iters):
            c=F.softmax(b,dim=1)
            s=(c.unsqueeze(-1)*votes).sum(dim=2)
            v=squash(s)
            agreement=(votes*v.unsqueeze(2)).sum(dim=-1)
            b=b+agreement
        lengths=torch.sqrt((v*v).sum(dim=-1)+1e-8)
        return lengths

def train_neurocaps(Xtr,ytr,Xva,yva,epochs=35,batch_size=32,lr=1e-3):
    model=NeuroCapsNet(Xtr.shape[1]).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    # margin loss for capsule lengths
    def margin_loss(out,y):
        one=F.one_hot(y,2).float()
        pos=F.relu(.9-out).pow(2); neg=F.relu(out-.1).pow(2)
        return (one*pos + .5*(1-one)*neg).sum(dim=1).mean()
    ds=TensorDataset(torch.tensor(Xtr,dtype=torch.float32),torch.tensor(ytr,dtype=torch.long))
    dl=DataLoader(ds,batch_size=min(batch_size,len(ds)),shuffle=True)
    history={"loss":[],"val_loss":[],"accuracy":[],"val_accuracy":[]}
    best=float("inf"); best_state=None; patience=7; wait=0
    for ep in range(epochs):
        model.train(); losses=[]; correct=0; total=0
        for xb,yb in dl:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(); out=model(xb); loss=margin_loss(out,yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),2.0); opt.step()
            losses.append(loss.item()); correct+=(out.argmax(1)==yb).sum().item(); total+=len(yb)
        model.eval()
        with torch.no_grad():
            ov=model(torch.tensor(Xva,dtype=torch.float32).to(DEVICE))
            vy=torch.tensor(yva,dtype=torch.long).to(DEVICE)
            vl=margin_loss(ov,vy).item(); vacc=(ov.argmax(1)==vy).float().mean().item()
        history["loss"].append(float(np.mean(losses)))
        history["val_loss"].append(vl)
        history["accuracy"].append(correct/max(total,1))
        history["val_accuracy"].append(vacc)
        if vl<best-1e-4:
            best=vl; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: break
    if best_state: model.load_state_dict(best_state)
    return model,history

def predict_prob(model,X):
    model.eval()
    with torch.no_grad():
        z=model(torch.tensor(X,dtype=torch.float32).to(DEVICE))
        p=F.softmax(z,dim=1)[:,1].cpu().numpy()
    return p

## 7. Leakage-controlled subject-wise five-fold evaluation

For each fold, the training partition alone fits imputation, Box–Cox, scaling, I-LDA, AGC-LSTM, feature selection and NeuroCapsNet. The held-out participants are transformed using the frozen training-fold components. A group-aware internal validation subset is used for early stopping.

In [ ]:
def safe_group_folds(X,y,groups,n_splits=5):
    unique_groups=np.unique(groups)
    if len(unique_groups)<n_splits:
        raise ValueError(f"Only {len(unique_groups)} independent groups; cannot run {n_splits}-fold subject-wise CV.")
    splitter=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=SEED)
    return list(splitter.split(X,y,groups))

def run_fold(X,y,groups,fold_id,train_idx,test_idx,
             agc_epochs=12,rco_population=12,rco_iterations=15,net_epochs=35):
    Xtr_raw,Xte_raw=X[train_idx],X[test_idx]
    ytr,yte=y[train_idx],y[test_idx]
    gtr=groups[train_idx]
    imputer=SimpleImputer(strategy="mean")
    A=imputer.fit_transform(Xtr_raw); B=imputer.transform(Xte_raw)
    box=ModifiedBoxCox().fit(A)
    A=box.transform(A); B=box.transform(B)
    scaler=StandardScaler().fit(A)
    A=scaler.transform(A).astype(np.float32); B=scaler.transform(B).astype(np.float32)
    lda=ImprovedLDA(n_components=1,max_features=min(256,A.shape[1])).fit(A,ytr)
    A=lda.transform(A); B=lda.transform(B)
    # handcrafted representation; correlation summary uses training-only correlation matrix
    Htr=handcrafted_features(A); Hte=handcrafted_features(B)
    Gtr,agc=train_agc(A,ytr,A,epochs=agc_epochs)
    Gte,_=train_agc_transform(agc,A,B) if False else (None,None)
    # Apply the exact trained AGC-LSTM model to held-out rows
    agc.eval()
    with torch.no_grad():
        Gte=agc(torch.tensor(B,dtype=torch.float32).to(DEVICE)).cpu().numpy()
    # Training label Gini is constant and does not leak test labels
    Ftr=np.concatenate([Htr,Gtr,gini_feature(ytr,len(A))],axis=1)
    Fte=np.concatenate([Hte,Gte,gini_feature(ytr,len(B))],axis=1)
    fs=StandardScaler().fit(Ftr)
    Ftr=fs.transform(Ftr).astype(np.float32); Fte=fs.transform(Fte).astype(np.float32)
    selected,curve=rco_select(Ftr,ytr,gtr,population=rco_population,iterations=rco_iterations,seed=SEED+fold_id)
    Ftr=Ftr[:,selected]; Fte=Fte[:,selected]
    # group-aware early-stopping validation split from training fold
    try:
        gi=GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED+fold_id)
        it,iv=next(gi.split(Ftr,ytr,gtr))
        if len(np.unique(ytr[it]))<2 or len(np.unique(ytr[iv]))<2: raise ValueError()
    except Exception:
        from sklearn.model_selection import train_test_split
        it,iv=train_test_split(np.arange(len(ytr)),test_size=.20,random_state=SEED,stratify=ytr)
    model,hist=train_neurocaps(Ftr[it],ytr[it],Ftr[iv],ytr[iv],epochs=net_epochs)
    prob=predict_prob(model,Fte); pred=(prob>=.5).astype(int)
    metrics={
        "fold":fold_id,"accuracy":accuracy_score(yte,pred),
        "precision":precision_score(yte,pred,zero_division=0),
        "recall":recall_score(yte,pred,zero_division=0),
        "f1":f1_score(yte,pred,zero_division=0),
        "specificity": (confusion_matrix(yte,pred,labels=[0,1])[0,0] /
                        max(confusion_matrix(yte,pred,labels=[0,1])[0].sum(),1)),
        "auc":roc_auc_score(yte,prob) if len(np.unique(yte))==2 else np.nan,
        "selected_features":len(selected)
    }
    return metrics, (yte,pred,prob,hist,curve,selected,model)

In [ ]:
# Complete the fold runner with the trained AGC-LSTM transform.
# (Kept separate to make the train-only fitting boundary explicit.)
results={}
fold_artifacts={}
for ds_name,(X,y,groups,feature_names,raw_df) in datasets.items():
    print("\n"+"="*70+"\n",ds_name)
    folds=safe_group_folds(X,y,groups,n_splits=5)
    rows=[]
    for f,(tr,te) in enumerate(folds,1):
        print(f"Fold {f}/5 | train={len(tr)} test={len(te)}")
        m,a=run_fold(X,y,groups,f,tr,te,agc_epochs=10,rco_population=10,rco_iterations=10,net_epochs=30)
        rows.append(m); fold_artifacts[(ds_name,f)]=a
        print({k:round(v,4) if isinstance(v,(float,np.floating)) else v for k,v in m.items()})
    res=pd.DataFrame(rows)
    results[ds_name]=res
    print("\nFold results:\n",res.to_string(index=False))
    print("\nMean ± SD:\n",res.drop(columns="fold").agg(["mean","std"]).T)

## 8. Aggregate metrics, confusion matrix and learning curves

In [ ]:
summary=[]
for name,df in results.items():
    row={"Dataset":name}
    for col in ["accuracy","precision","recall","specificity","f1","auc","selected_features"]:
        row[col+"_mean"]=df[col].mean()
        row[col+"_sd"]=df[col].std()
    summary.append(row)
summary_df=pd.DataFrame(summary)
display(summary_df.round(4))
summary_df.to_csv("RCO_NeuroCapsNet_CV_Summary.csv",index=False)

for name in results:
    fig,ax=plt.subplots(figsize=(7,4))
    ax.bar(results[name]["fold"],results[name]["accuracy"]*100)
    ax.set_title(f"{name} — Subject-wise fold accuracy")
    ax.set_xlabel("Fold"); ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0,100)
    ax.grid(axis="y",alpha=.25); plt.show()

# Display training history for the first dataset's first fold.
first_name=list(datasets)[0]
hist=fold_artifacts[(first_name,1)][3]
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].plot(hist["loss"],label="Training"); axs[0].plot(hist["val_loss"],label="Validation")
axs[0].set_title("NeuroCapsNet loss"); axs[0].set_xlabel("Epoch"); axs[0].legend()
axs[1].plot(np.array(hist["accuracy"])*100,label="Training")
axs[1].plot(np.array(hist["val_accuracy"])*100,label="Validation")
axs[1].set_title("NeuroCapsNet accuracy"); axs[1].set_xlabel("Epoch"); axs[1].set_ylabel("%"); axs[1].legend()
plt.tight_layout(); plt.show()

## 9. Explainability: permutation importance and feature attribution

The manuscript discusses SHAP and attention interpretation. This notebook provides a model-agnostic permutation-importance baseline over the selected feature vector for a held-out fold. The original raw feature names cannot be mapped one-to-one after I-LDA and engineered representation fusion; therefore these scores are representation-level, not named acoustic biomarkers. Do not label them as SHAP values.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer, accuracy_score

name=list(datasets)[0]
yte,pred,prob,hist,curve,selected,model=fold_artifacts[(name,1)]
print("Fold-1 held-out report:")
print(classification_report(yte,pred,target_names=["Healthy/control","Parkinson's disease"],zero_division=0))
cm=confusion_matrix(yte,pred,labels=[0,1])
plt.figure(figsize=(5,4)); plt.imshow(cm)
plt.title(f"{name} — held-out fold confusion matrix")
plt.xticks([0,1],["Control","PD"]); plt.yticks([0,1],["Control","PD"])
for i in range(2):
    for j in range(2): plt.text(j,i,str(cm[i,j]),ha="center",va="center")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.colorbar(); plt.show()

## 10. Save results and reproducibility metadata

In [ ]:
import json, platform
metadata={
    "framework":"RCO-Enhanced NeuroCapsNet",
    "seed":SEED,
    "device":str(DEVICE),
    "python":platform.python_version(),
    "datasets":{k:{"rows":int(len(v[1])),"features":int(v[0].shape[1]),
                   "groups":int(len(np.unique(v[2])))} for k,v in datasets.items()},
    "validation":"StratifiedGroupKFold, 5 folds; participant grouped",
    "notes":[
        "DB1 and DB2 are duplicate Oxford dataset distributions per manuscript.",
        "No performance values are hard-coded; metrics are calculated from loaded CSVs.",
        "If a reliable participant identifier is unavailable, subject-independent validation is not established."
    ]
}
with open("RCO_NeuroCapsNet_metadata.json","w") as f: json.dump(metadata,f,indent=2)
for name,df in results.items():
    df.to_csv(f"{name}_fold_metrics.csv",index=False)
print("Saved summary, per-fold metrics and metadata in the current directory.")

## 11. Optional: export files from Google Colab

In [ ]:
try:
    from google.colab import files
    files.download("RCO_NeuroCapsNet_CV_Summary.csv")
except Exception:
    print("Files are saved in the notebook working directory.")